# SAE / VUF MUC (Colab)

Пайплайн `sae_muc.run_muc` с двумя режимами интервенции:

- **`sae`** — латентный bump: **encode → f + αδ → decode + error** (нужен `mistral_intervention.pt` из `build_intervention_config`).
- **`residual`** — корректировка как в статье: **h ← h + α·r̂** по строкам `Hs_hedge` (сырой остаток, без SAE на инференсе).

**Данные в `REPO_DIR`** (по умолчанию `/content/sae-muc`):

- `datasets/{dataset}/{model}/{split}.csv`
- `detection/LR_outputs/{dataset}/{model}/{split}_verbal_uncertainty_sentence_semantic_entropy.json`
- `Hs_hedge_universal.pt` — для **residual** и для сборки SAE-конфига (см. §6–§7, путь задаётся в ячейке).

**Простой старт:** §2 — загрузка **одного zip** с папками `datasets/`, `detection/`, `calibration/` в корне архива. Либо скопируйте с Drive (§3–§4).

**Dry run:** в §7 задайте `DRY_RUN_N = 5` (первые N вопросов); для полного прогона — `None`.

**Нужны:** GPU, Hugging Face токен для Mistral. Если сессия «засорена» импортами: **Runtime → Restart runtime**, затем ячейки по порядку.

## 0. GPU (без `import torch`)

**Важно:** не импортируйте `torch` до ячейки с `pip` — иначе в процессе залипает старый NumPy и дальше `transformers` падает с `dtype size changed`.

In [ ]:
import subprocess

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

## 1. Клон репозитория и зависимости

**Время:** клон — секунды; **pip** (особенно **transformers** с `--force-reinstall`) — нередко **3–10 минут**. Лог pip в ячейке полный, без `-q`.

In [ ]:
import os, sys, subprocess

GIT_URL = os.environ.get("SAE_MUC_GIT_URL", "https://github.com/SadreevAmir/sae-muc.git")
GIT_BRANCH = os.environ.get("SAE_MUC_BRANCH", "main")
REPO_DIR = os.environ.get("SAE_MUC_REPO_DIR", "/content/sae-muc")

sae_pkg = os.path.join(REPO_DIR, "sae_muc")
if not os.path.isdir(sae_pkg):
    parent = os.path.dirname(REPO_DIR.rstrip("/")) or "/content"
    os.makedirs(parent, exist_ok=True)
    if os.path.isdir(REPO_DIR):
        subprocess.run(["rm", "-rf", REPO_DIR], check=True)
    subprocess.run(
        ["git", "clone", "--depth", "1", "-b", GIT_BRANCH, GIT_URL, REPO_DIR],
        check=True,
    )
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)

assert os.path.isdir(sae_pkg), f"После клона ожидается {sae_pkg}"
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# PyTorch не трогаем. Сброс NumPy + совместимые бинари transformers (иначе dtype size changed).
print("→ pip uninstall numpy …")
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "numpy"])
print("→ pip install numpy (force-reinstall) …")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--no-cache-dir", "-U", "--force-reinstall",
    "numpy>=2.0.0,<2.1",
])
print("→ pip install transformers + accelerate (force-reinstall, может быть долго) …")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-U", "--no-cache-dir", "--force-reinstall",
    "transformers>=4.40", "accelerate",
])
print("→ pip install sae-lens и остальное …")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-U",
    "sae-lens>=6.0", "pandas", "tqdm", "jsonlines", "huggingface_hub",
])
subprocess.check_call([
    sys.executable, "-c",
    "import numpy, numpy.random; print('numpy OK', numpy.__version__)",
])

import torch

assert torch.cuda.is_available(), "Runtime → тип подключения: GPU"
print("REPO_DIR:", REPO_DIR)
print("GPU:", torch.cuda.get_device_name(0))


## 2. Данные VUF: один zip → `REPO_DIR`

Соберите архив так, чтобы **в корне zip** были каталоги `datasets/`, `detection/`, `calibration/` (как в чекпоинте фаз 1–6). Включите файл **`Hs_hedge_universal.pt`** (часто лежит под `calibration/outputs/.../uncertainty/`).

Установите `DO_UPLOAD_ZIP = True` и выполните ячейку — выберите файл. Если данные уже на месте (Drive, предыдущий запуск), поставьте `False`.

In [ ]:
import os
import zipfile

REPO_DIR = os.environ.get("SAE_MUC_REPO_DIR", "/content/sae-muc")

# True — диалог загрузки zip в Colab
DO_UPLOAD_ZIP = False

if DO_UPLOAD_ZIP:
    try:
        from google.colab import files
        print("Загрузите zip (в корне: datasets/, detection/, calibration/) …")
        uploaded = files.upload()
        for fn in uploaded:
            if str(fn).lower().endswith(".zip"):
                with zipfile.ZipFile(fn, "r") as z:
                    z.extractall(REPO_DIR)
                print("Распаковано в", REPO_DIR)
                break
        else:
            print("Нет .zip среди загруженных файлов.")
    except ImportError:
        print("Не Colab — распакуйте данные в REPO_DIR вручную.")
else:
    print("DO_UPLOAD_ZIP=False — ожидаются данные уже в REPO_DIR.")

## 3. Google Drive (опционально): монтирование и бэкап

Поставьте `MOUNT_DRIVE = False`, если данные уже в `REPO_DIR` и бэкап не нужен. Иначе смонтируйте Drive и задайте `DRIVE_BACKUP_ROOT`; каждые `BACKUP_INTERVAL_SEC` секунд копируются `sae_muc/outputs` и `sae_muc/artifacts`.

In [ ]:
import os
import sys
from pathlib import Path

REPO_DIR = os.environ.get("SAE_MUC_REPO_DIR", "/content/sae-muc")
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

MOUNT_DRIVE = False  # True — смонтировать Drive и фоновый бэкап
DRIVE_BACKUP_ROOT = "/content/drive/MyDrive/vuf_sae_muc_backup"
BACKUP_INTERVAL_SEC = 600

_drive_backup = None
if MOUNT_DRIVE:
    try:
        from google.colab import drive
        from sae_muc.drive_sync import PeriodicDriveBackup

        drive.mount("/content/drive")
        pairs = [
            (f"{REPO_DIR}/sae_muc/outputs", f"{DRIVE_BACKUP_ROOT}/sae_muc/outputs"),
            (f"{REPO_DIR}/sae_muc/artifacts", f"{DRIVE_BACKUP_ROOT}/sae_muc/artifacts"),
        ]
        Path(DRIVE_BACKUP_ROOT).mkdir(parents=True, exist_ok=True)
        _drive_backup = PeriodicDriveBackup(pairs, interval_sec=BACKUP_INTERVAL_SEC)
        _drive_backup.start()
        print("Периодический бэкап на Drive запущен. Остановка: _drive_backup.stop()")
    except ImportError:
        print("Не Colab — пропуск Drive.")
else:
    print("MOUNT_DRIVE=False — без монтирования Drive.")

## 4. (Опционально) Чекпоинт фаз 1–6 с Drive → `REPO_DIR`

Если данные не заливали zip в §2, а лежат на Drive в папке вроде `vuf_checkpoint_phase6`, скопируйте `datasets`, `detection`, `calibration` в `REPO_DIR`.

In [ ]:
import os, shutil
from pathlib import Path

REPO_DIR = Path(os.environ.get("SAE_MUC_REPO_DIR", "/content/sae-muc"))
CHECKPOINT_SRC = Path("/content/drive/MyDrive/vuf_checkpoint_phase6")  # или None

if CHECKPOINT_SRC and CHECKPOINT_SRC.is_dir():
    for name in ("datasets", "detection", "calibration"):
        src = CHECKPOINT_SRC / name
        dst = REPO_DIR / name
        if src.is_dir():
            shutil.copytree(src, dst, dirs_exist_ok=True)
            print("Copied", name)
else:
    print("Пропуск копирования чекпоинта (нет CHECKPOINT_SRC).")

## 5. Hugging Face login

In [ ]:
from huggingface_hub import login
login()  # токен с доступом к Mistral

## 6. Конфиг: режим, dry run, путь к `Hs_hedge`, сборка `intervention.pt`

**`STEERING`**: `"sae"` или `"residual"` (**h ← h + α·r̂**). **`DRY_RUN_N`**: число или `None` для всего сплита.

**Классический VUF и слои:** при `residual` вектор добавляется **только на выбранные HF-слои**. Если **`VUF_LAYERS = None`**, список берётся как пересечение `STR_PROCESS_LAYERS` с слоями из **`mistral_intervention.pt`** (если файл уже есть после прогона SAE) или с релизом **`VUF_ALIGN_RELEASE`** (как у SAE на Mistral: слои 7, 15, 23 — в `range(15,32)` остаются 15 и 23, как у SAE). Явно задайте **`VUF_LAYERS = "15,23"`**, если нужно зафиксировать слои вручную.

Сборка **`mistral_intervention.pt`** — только при `STEERING == "sae"`. `Hs_hedge` — список **`HEDGE_CANDIDATES`**.

**`PROMPT_TYPE`**: `"plain"` — только user `Question: …\nAnswer:` **без** системного текста про hedging; `"uncertainty"` — как в Meta MUC (hedge-инструкция в system). Для согласованности с `merged/.../uncertainty/Hs_hedge` обычно нужен **`uncertainty`**; `plain` — если осознанно меняете постановку (имя jsonl получит суффикс `_plain`).

In [ ]:
import os
import subprocess
import sys

REPO_DIR = os.environ.get("SAE_MUC_REPO_DIR", "/content/sae-muc")

# "sae" | "residual"
STEERING = "sae"
# Тест на первых N строках CSV; None — весь split
DRY_RUN_N = 5

# Какие слои участвуют в α / детекции (как в Meta); SAE вешается только на подмножество (есть SAE)
STR_PROCESS_LAYERS = "range(15,32)"

# "uncertainty" | "plain" | "sentence" — см. текст §6
PROMPT_TYPE = "plain"

# Только residual: None → авто (пересечение с SAE-слоями); или явно "15,23" / "range(15,32)"
VUF_LAYERS = None
VUF_ALIGN_RELEASE = "mistral-7b-res-wg"  # если нет intervention.pt, откуда брать список HF-слоёв SAE

MN = "Mistral-7B-Instruct-v0.3"
_merged_unc = f"{REPO_DIR}/calibration/outputs/merged/{MN}/uncertainty/Hs_hedge_universal.pt"
_merged_sent = f"{REPO_DIR}/calibration/outputs/merged/{MN}/sentence/Hs_hedge_universal.pt"
_nq_unc = f"{REPO_DIR}/calibration/outputs/nq_open/{MN}/uncertainty/Hs_hedge_universal.pt"
if PROMPT_TYPE == "sentence":
    HEDGE_CANDIDATES = [_merged_sent, _merged_unc, _nq_unc]
else:
    HEDGE_CANDIDATES = [_merged_unc, _merged_sent, _nq_unc]
HEDGE_PATH = next((p for p in HEDGE_CANDIDATES if os.path.isfile(p)), None)
if HEDGE_PATH is None:
    raise FileNotFoundError(
        "Не найден Hs_hedge_universal.pt. Проверьте §2/§4 или добавьте путь в HEDGE_CANDIDATES."
    )
print("HEDGE_PATH:", HEDGE_PATH)

OUT_INTERVENTION = f"{REPO_DIR}/sae_muc/artifacts/mistral_intervention.pt"

if STEERING == "sae":
    os.makedirs(os.path.dirname(OUT_INTERVENTION), exist_ok=True)
    cmd = [
        sys.executable,
        "-m",
        "sae_muc.build_intervention_config",
        "--hedge_path",
        HEDGE_PATH,
        "--out_path",
        OUT_INTERVENTION,
        "--release",
        "mistral-7b-res-wg",
        "--top_k",
        "64",
        "--sae_device",
        "cpu",
    ]
    subprocess.check_call(cmd, cwd=REPO_DIR)
    print("Собрано:", OUT_INTERVENTION)
else:
    print('STEERING="residual" — сборка intervention.pt не нужна.')

## 7. Запуск `run_muc`

Параметры из §6. Имя jsonl: полный **sae**, `uncertainty`, без dry run — `with_vufi_2_range(15,32)_1.0.jsonl`; при **`PROMPT_TYPE != "uncertainty"`** к имени добавляется `_plain` / `_sentence`; **residual** — ещё `_residual_L…`; dry run — `_firstN`.

Ячейка ниже делает **`git pull`**, проверяет, что в `run_muc.py` есть актуальные флаги, и запускает **`main()` в том же процессе**, что и ядро (не через subprocess) — так не теряется лог и не путаются версии Python.

Если проверка падает с **«нет --steering»**: залейте актуальный `sae-muc` на GitHub и снова выполните ячейку (или замените `sae_muc/run_muc.py` вручную в Colab).

In [ ]:
import os
import subprocess
import sys

REPO_DIR = os.environ.get("SAE_MUC_REPO_DIR", "/content/sae-muc")
# Раскомментируйте, чтобы писать jsonl в дерево calibration (как Meta eval):
OUTPUT_DIR = None  # f"{REPO_DIR}/calibration/outputs/nq_open/Mistral-7B-Instruct-v0.3/uncertainty/test"

cmd = [
    sys.executable,
    "-m",
    "sae_muc.run_muc",
    "--repo_root",
    REPO_DIR,
    "--dataset",
    "nq_open",
    "--split",
    "test",
    "--model_name",
    "Mistral-7B-Instruct-v0.3",
    "--prompt_type",
    PROMPT_TYPE,
    "--str_process_layers",
    STR_PROCESS_LAYERS,
    "--steering",
    STEERING,
    "--max_alpha",
    "1.0",
]
if STEERING == "sae":
    cmd += ["--intervention_path", OUT_INTERVENTION]
else:
    cmd += ["--hedge_path", HEDGE_PATH, "--intervention_path", OUT_INTERVENTION]
    cmd += ["--vuf_align_release", VUF_ALIGN_RELEASE]
    if VUF_LAYERS:
        cmd += ["--vuf_layers", VUF_LAYERS]

if DRY_RUN_N is not None:
    cmd += ["--max_questions", str(int(DRY_RUN_N))]

if OUTPUT_DIR:
    cmd += ["--output_dir", OUTPUT_DIR]

print("Эквивалент CLI:", " ".join(cmd))

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

_br = os.environ.get("SAE_MUC_BRANCH", "main")
_gp = subprocess.run(
    ["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", _br],
    capture_output=True,
    text=True,
)
if _gp.stdout:
    print(_gp.stdout, end="")
if _gp.stderr:
    print(_gp.stderr, end="", file=sys.stderr)

_rmp = os.path.join(REPO_DIR, "sae_muc", "run_muc.py")
with open(_rmp, encoding="utf-8") as _f:
    _src = _f.read()
if "--steering" not in _src or "max_questions" not in _src:
    raise RuntimeError(
        f"В {_rmp} старая версия (нет CLI --steering / --max_questions). "
        "Сделайте push актуального репозитория на GitHub и повторите ячейку, "
        "или вручную замените sae_muc/*.py в Colab."
    )

for _name in list(sys.modules):
    if _name == "sae_muc" or _name.startswith("sae_muc."):
        del sys.modules[_name]

from sae_muc.run_muc import main

_cli = cmd[3:]
_saved_argv = sys.argv
sys.argv = ["sae_muc.run_muc"] + _cli
try:
    main()
finally:
    sys.argv = _saved_argv

## 8. Остановить бэкап (опционально)

In [ ]:
# if _drive_backup is not None:
#     _drive_backup.stop()